In [3]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [4]:
#### 2. Preparação dos Dados (Split Treino/Teste e Estandardização)
import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Carregar o dataset tratado gerado pelo primeiro notebook
df_clean = pd.read_csv('../data/dataset_trabalhado.csv')

# 1. Separar as características (X) do preço (y)
X = df_clean.drop(columns=["price_eur", "id"], errors="ignore")
y = df_clean["price_eur"]

# 2. Definir grupos de colunas (numéricas, binárias e categóricas dummy)
num_cols = [
    "area_m2", "latitude", "longitude",
    "bedrooms", "bathrooms", "floor", "num_floors"
]
bin_cols = ["has_garage", "has_garden"]
cat_cols = [c for c in X.columns if c.startswith("property_type_")]

# 3. Dividir em 80% treino e 20% teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 4. Criar o ColumnTransformer (estandardizar numéricas e manter binárias/dummies)
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("bin", "passthrough", bin_cols + cat_cols),
    ]
)

# 5. Ajustar apenas no treino e transformar treino e teste
X_train_scaled = preprocessor.fit_transform(X_train)
X_test_scaled = preprocessor.transform(X_test)

# 6. Guardar o preprocessor para transformar imóveis novos no futuro
joblib.dump(preprocessor, "preprocessor.joblib")

print("Dimensão de X_train pronto:", X_train_scaled.shape)
print("Exemplo de preços intactos (y_train):", y_train.head(3).values)
print(f"Treino: {len(X_train)} imóveis | Teste: {len(X_test)} imóveis")

Dimensão de X_train pronto: (5399, 11)
Exemplo de preços intactos (y_train): [250000 560000 250000]
Treino: 5399 imóveis | Teste: 1350 imóveis


In [5]:
#### 3. Treino e Avaliação do Modelo
import joblib
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# 1. Instanciar e treinar (fit) o modelo com os dados de treino estandardizados
modelo = HistGradientBoostingRegressor(random_state=42)
modelo.fit(X_train_scaled, y_train)

# 2. Fazer previsões e avaliar no conjunto de teste
y_pred = modelo.predict(X_test_scaled)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

# 3. Guardar o modelo treinado
joblib.dump(modelo, "modelo_imoveis.joblib")

print(f"MAE (Erro Médio Absoluto): {mae:,.2f} €")
print(f"R² Score: {r2:.4f}")

MAE (Erro Médio Absoluto): 133,341.74 €
R² Score: 0.7424
